# Rule Swing Label Build Guide

Purpose:
- convert the cleaned 15-minute OHLC dataset into rule-based swing labels for ML experiments

Inputs:
- `ETHUSDT_15m_ohlc_clean.parquet`
- `swing_highs_lows_online(...)` from the rule-based detector module

Output:
- `swing_labels.parquet` with `y_high_rule` and `y_low_rule`

Reading guide:
1. Load the cleaned candles.
2. Run the rule detector on `high`, `low`, and `close`.
3. Convert detector events into 0/1 labels aligned to each bar.
4. Inspect the label counts before saving.

Interpretation note:
- These labels describe historical swing structure for supervised learning; they are not direct trading signals on their own.


### Imports + path configuration


In [ ]:
# Centralize folder locations here so you only update paths once after moving files.

import sys
from pathlib import Path

import pandas as pd

def locate_ml_root(start=None) -> Path:
    start = (Path.cwd() if start is None else Path(start)).resolve()
    for candidate in [start] + list(start.parents):
        if candidate.name == "ML v1" and (candidate / "data").exists():
            return candidate

        ml_root = candidate / "ML v1"
        if (ml_root / "data").exists() and (ml_root / "code").exists():
            return ml_root

    raise FileNotFoundError(
        "Could not locate the 'ML v1' workspace from the current working directory."
    )


ML_ROOT = locate_ml_root()
PROJECT_ROOT = ML_ROOT.parent
DATA_DIR = ML_ROOT / "data"
CONFIG_DIR = ML_ROOT / "configs"
CODE_DIR = ML_ROOT / "code"

ALGO_DIR = PROJECT_ROOT / "Algorythm v1"
OHLC_CLEAN_PATH = DATA_DIR / "ETHUSDT_15m_ohlc_clean.parquet"
SWING_LABELS_PATH = DATA_DIR / "swing_labels.parquet"

if str(ALGO_DIR) not in sys.path:
    sys.path.insert(0, str(ALGO_DIR))

from swing_high_low_detection import swing_highs_lows_online


### Load CSV (safe) + quick preview

In [ ]:
ohlc_clean = pd.read_parquet(OHLC_CLEAN_PATH)

ohlc_clean.head()


### Run the swing detection
#### Produces swings with 'HighLow' = 1 for highs, -1 for lows, NaN otherwise

label_type: rule_swing
N_candidates: [5, 10, 20, 50]
N_confirmation: 3
min_move_threshold: 0.0
min_bars_between_swings: 3
uses_future: true
computed_on: ETHUSDT_15m_ohlc_clean.parquet


In [3]:
# Only the price columns needed by the detector are passed in, which keeps the label definition explicit.

swings_df = swing_highs_lows_online(ohlc_clean[["high", "low", "close"]])


### Build labels DataFrame
#### 0/1 encoding for ML: 1 = swing, 0 = no swing

In [4]:
# Keep timestamp and segment_id so downstream joins with engineered features stay bar-aligned.

labels_df = pd.DataFrame({
    "timestamp": ohlc_clean["timestamp"],
    "segment_id": ohlc_clean["segment_id"],
    "y_high_rule": (swings_df["HighLow"] == 1).astype(int),
    "y_low_rule":  (swings_df["HighLow"] == -1).astype(int),
})


### Quick sanity check: how many swings were detected and how df looks like

In [5]:
print(labels_df[["y_high_rule", "y_low_rule"]].sum())

y_high_rule    16358
y_low_rule     15810
dtype: int64


In [6]:
labels_df.head()

,timestamp,segment_id,y_high_rule,y_low_rule
0,2021-07-05 12:00:00+00:00,0,1,0
1,2021-07-05 12:15:00+00:00,0,0,0
2,2021-07-05 12:30:00+00:00,0,0,0
3,2021-07-05 12:45:00+00:00,0,0,0
4,2021-07-05 13:00:00+00:00,0,1,0


### Save labels as a separate file

In [ ]:
# Save the labels as a standalone artifact in the centralized data folder.

labels_df.to_parquet(SWING_LABELS_PATH, index=False)
